### **Import necessary libraries**

In [ ]:
import os 
import sys 

import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
%matplotlib inline

In [ ]:
current_cwd = os.getcwd()
current_cwd
new_cwd = '\\'.join(current_cwd.split('\\')[:2])
sys.path.append(new_cwd + '\\utils')

In [ ]:
from helper import *

### **Load the dataset**

In [ ]:
%%capture
from datasets import load_dataset

dataset = load_dataset("ncduy/mt-en-vi")

In [ ]:
dataset

In [ ]:
train = dataset['train'].to_pandas()
val = dataset['validation'].to_pandas()
test = dataset['test'].to_pandas()

In [ ]:
envi_data = pd.concat(
    [
        train, 
        val,
        test
    ]
)
envi_data.reset_index(drop=True, inplace=True)

In [ ]:
envi_data.head()

In [ ]:
envi_data.tail()

In [ ]:
envi_data.shape

### **Filter dataset**

We want to filter the dataset to include only rows where both the English and Vietnamese sentences have a maximum length of 50 words, normalize sentence and contain no special characters.

In [ ]:
# Normalize the sentences in both columns
envi_data["en"] = envi_data["en"].apply(normalize_sentence)
envi_data["vi"] = envi_data["vi"].apply(normalize_sentence)

# Filter the DataFrame: English sentences should have at most 50 words and be all ASCII,
# while Vietnamese sentences should have at most 50 words and only allowed characters.
filtered_envi_data = envi_data[
    (envi_data["en"].apply(lambda s: len(s.split()) <= 50 and s.isascii()))
    & (envi_data["vi"].apply(lambda s: len(s.split()) <= 50 and is_valid_vietnamese(s)))
].reset_index(drop=True)

# Check the shape of the filtered dataset
print(filtered_envi_data.shape)


### **Randomly select 300k (or 500k) rows from the filtered dataset**

In [ ]:
import pandas as pd

df = filtered_envi_data.sample(n=500000, random_state=42).reset_index(drop=True)

In [ ]:
print(df.shape)

In [ ]:
cnt = 0
# cnt = 100000
# cnt = 200000
# cnt = 300000
# cnt = 400000

while cnt < 100000:
# while cnt < 200000:
# while cnt < 300000:
# while cnt < 400000:
# while cnt < df.shape[0]:
    df.loc[cnt, ['en']] = df.loc[cnt, ['en']].map(spell_correction)
    df.iloc[cnt, :] = df.iloc[cnt, :].map(tokenize_and_join)
    if cnt % 100 == 0:
        print(cnt)
    cnt += 1

print("Complete map the tokenize_and_join function")

This preprocessing step iterates through each row of the DataFrame, applying two main functions to clean and standardize the text data. First, the `tokenize_and_join` function is applied across all columns to tokenize and then rejoin the text, ensuring a consistent format. Second, the `spell_correction` function is specifically applied to the `"en"` column to correct any misspelled words.

In [ ]:
df_part1 = df.iloc[:100000]
# df_part2 = df.iloc[100000:200000]
# df_part3 = df.iloc[200000:300000]
# df_part4 = df.iloc[300000:400000]
# df_part5 = df.iloc[400000:]

In [ ]:
# df_part1.to_csv('dataset_part1.csv', index=False)
# df_part2.to_csv('dataset_part2.csv', index=False)
# df_part3.to_csv('dataset_part3.csv', index=False)
# df_part4.to_csv('dataset_part4.csv', index=False)
# df_part5.to_csv('dataset_part5.csv', index=False)

In [ ]:
envi_data = pd.concat(
    [
        pd.read_csv(f"{new_cwd}/data/dataset_part1.csv"),
        pd.read_csv(f"{new_cwd}/data/dataset_part2.csv"),
        pd.read_csv(f"{new_cwd}/data/dataset_part3.csv"),
    ]
)
envi_data.reset_index(drop=True, inplace=True)

envi_data.to_csv(f'{new_cwd}/data/dataset_300k.csv', index=False)


# envi_data = pd.concat(
#     [
#         pd.read_csv(f"{new_cwd}/data/dataset_part1.csv"),
#         pd.read_csv(f"{new_cwd}/data/dataset_part2.csv"),
#         pd.read_csv(f"{new_cwd}/data/dataset_part3.csv"),
#         pd.read_csv(f"{new_cwd}/data/dataset_part4.csv"),
#         pd.read_csv(f"{new_cwd}/data/dataset_part5.csv"),
#     ]
# )
# envi_data.reset_index(drop=True, inplace=True)

# envi_data.to_csv(f'{new_cwd}/data/dataset_500k.csv', index=False)